#**CHAT BOT**  

In [ ]:
#lIBRERIAS
import numpy as np
import math as math
import matplotlib.pyplot as plt
import re, os
#La siguiente línea evita la generación de "warnings" y "flags" al importar keras y tensorflow.
#Dichas advertencias no comprometen el funcionamiento del código.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
#Importación de keras y tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Flatten,Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Attention, Concatenate, Bidirectional



In [ ]:
#Carga de datos
import requests

import urllib.request
import gdown

url = "1hdorIG5a_v7NmKGfTuFPEJXDgBPznfEj"


url_quotes = "https://raw.githubusercontent.com/jhon1142/Proyecto_Deep_Learning/main/moviequotes.memorable_quotes.txt"
url_pairs = "https://raw.githubusercontent.com/jhon1142/Proyecto_Deep_Learning/main/moviequotes.memorable_nonmemorable_pairs.txt"

memorable_quotes = requests.get(url_quotes).text
pairs = requests.get(url_pairs).text
gdown.download(id=url, output="moviequotes.scripts.txt", quiet=False)

scripts = open("moviequotes.scripts.txt", encoding="latin-1").read()
#scripts = open("moviequotes.scripts.txt", encoding="latin-1").read()

Downloading...
From (original): https://drive.google.com/uc?id=1hdorIG5a_v7NmKGfTuFPEJXDgBPznfEj
From (redirected): https://drive.google.com/uc?id=1hdorIG5a_v7NmKGfTuFPEJXDgBPznfEj&confirm=t&uuid=1499391a-5133-462e-9242-a94176e30b70
To: /content/moviequotes.scripts.txt
100%|██████████| 126M/126M [00:01<00:00, 69.4MB/s]


In [ ]:
print(len(scripts))
print(scripts[:500])

126342398
0 +++$+++ "murderland" +++$+++ 1 +++$+++ announcer +++$+++  +++$+++ Ladies and gentlemen, the official mascot of Murderland.... Scraps the Dog !
1 +++$+++ "murderland" +++$+++ 2 +++$+++ announcer +++$+++  +++$+++ Choose the doorway that starts you on your magical journey into  MURDERLAND !
2 +++$+++ "murderland" +++$+++ 3 +++$+++ johnny +++$+++  +++$+++ I didn't think he'd make it past Scraps.
3 +++$+++ "murderland" +++$+++ 4 +++$+++ bruce +++$+++ 2 +++$+++ Let's just see if he can make it into 


## LIMPIEZA

Ahora hacemos proprocesamiento

In [ ]:
#moviequotes.memorable_quotes
import re

clean_pattern = re.compile(r'[^a-zA-Z0-9áéíóúüñÁÉÍÓÚÜÑ¿?\.\' ]')

def clean_text(text):
    text = clean_pattern.sub(' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_memorable_pairs(data):

    lines = [l.strip() for l in data.split("\n") if l.strip()]
    pairs = []

    i = 0
    while i < len(lines)-2:

        movie = lines[i]
        memorable = lines[i+1]
        nonmem = lines[i+2]

        # eliminar ID numérico
        nonmem = re.sub(r'^\d+\s+', '', nonmem)

        pairs.append((clean_text(nonmem), clean_text(memorable)))

        i += 3

    return pairs

  #moviequotes.scripts

def parse_scripts(data):

    lines = data.split("\n")
    dialogs = []

    for line in lines:

        parts = line.split(" +++$+++ ")

        if len(parts) == 6:
            text = parts[-1]
            dialogs.append(clean_text(text))

    return dialogs

# Crear pares de conversación
def build_dialog_pairs(dialogs):

    pairs = []

    for i in range(len(dialogs)-1):

        inp = dialogs[i]
        out = dialogs[i+1]

        if inp and out:
            pairs.append((inp, out))

    return pairs

In [ ]:
mem_pairs = parse_memorable_pairs(memorable_quotes)

dialogs = parse_scripts(scripts)

script_pairs = build_dialog_pairs(dialogs)

training_pairs = mem_pairs + script_pairs

In [ ]:
dialogs

['ladies and gentlemen the official mascot of murderland.... scraps the dog',
 'choose the doorway that starts you on your magical journey into murderland',
 "i didn't think he'd make it past scraps.",
 "let's just see if he can make it into round two....",
 "don't.",
 'what ?',
 "don't eat at the console.",
 'what ? are you my mother ?',
 "it's my control board and i don't want it acting squirrelly because you dropped a few crumbs into the keyboard no put the shit away.",
 'yes sir mom.',
 'have a seat mr. simpson murderland was created in 1987. taking the original 82nd street subway station which was abandoned in 1913 our engineers were able to construct all thirty rides in less than......',
 '..... all thirty rides and attractions our engineers have built.....',
 'never underestimate a man who cheats on his taxes. his file says he is an avid jogger.',
 "you've got to be kidding me his lazy ass couldn't win the special olympics.",
 'mr. simpson so many doors so many choices. which do

In [ ]:
script_pairs

[('ladies and gentlemen the official mascot of murderland.... scraps the dog',
  'choose the doorway that starts you on your magical journey into murderland'),
 ('choose the doorway that starts you on your magical journey into murderland',
  "i didn't think he'd make it past scraps."),
 ("i didn't think he'd make it past scraps.",
  "let's just see if he can make it into round two...."),
 ("let's just see if he can make it into round two....", "don't."),
 ("don't.", 'what ?'),
 ('what ?', "don't eat at the console."),
 ("don't eat at the console.", 'what ? are you my mother ?'),
 ('what ? are you my mother ?',
  "it's my control board and i don't want it acting squirrelly because you dropped a few crumbs into the keyboard no put the shit away."),
 ("it's my control board and i don't want it acting squirrelly because you dropped a few crumbs into the keyboard no put the shit away.",
  'yes sir mom.'),
 ('yes sir mom.',
  'have a seat mr. simpson murderland was created in 1987. taking th

In [ ]:
training_pairs

[('who knocked up your sister?', 'who knocked up your sister?'),
 ("i watched you out there i've never seen you look like that",
  "i was watching you out there before. i've never seen you look so sexy."),
 ("you're eighteen. you don't know what you want. you won't know until you're forty five and you don't have it.",
  "you're 18 you don't know what you want. and you won't know what you want 'til you're 45 and even if you get it you'll be too old to use it."),
 ("see that? who needs affection when i've got blind hatred?",
  'ooh see that there. who needs affection when i have blind hatred?'),
 ("just because you're beautiful doesn't mean you can treat people like they don't matter.",
  "just 'cause you're beautiful that doesn't mean that you can treat people like they don't matter."),
 ("you're asking me out. that's so cute. what's your name again?",
  "you're asking me out? that's so cute what's your name again?"),
 ("leave it to you to use big words when you're shitfaced.",
  "leave

Validacion estructura del texto

In [ ]:
#Longitud de las frases de entrada
input_lengths = [len(pair[0].split()) for pair in training_pairs]

print("max:", max(input_lengths))
print("min:", min(input_lengths))
print("promedio:", sum(input_lengths)/len(input_lengths))

max: 1920
min: 1
promedio: 11.741832622385628


Se realiza una Distribución de longitudes para establer percentiles en la longitud de los textos y disminuir el ruido

In [ ]:
print("p90:", np.percentile(input_lengths, 90))
print("p95:", np.percentile(input_lengths, 95))
print("p99:", np.percentile(input_lengths, 99))

p90: 26.0
p95: 36.0
p99: 68.0


Esto significa:

- 90 % de las frases tienen ≤ 26 palabras

- 95 % tienen ≤ 36

- el 1 % restante es ruido o monólogos largos

Con esos percentiles ya podemos decidir un límite razonable de longitud

In [ ]:
MAX_LEN = 25

filtered_pairs = [
    (inp, out)
    for inp, out in training_pairs
    if len(inp.split()) <= MAX_LEN and len(out.split()) <= MAX_LEN
]
training_pairs = filtered_pairs

In [ ]:
#Verificar longitud real en training_pairs
max_input = max(len(inp.split()) for inp, _ in training_pairs)
max_output = max(len(out.split()) for _, out in training_pairs)

print("max input:", max_input)
print("max output:", max_output)

max input: 25
max output: 25


## separar inputs / outputs

In [ ]:
input_texts = [inp for inp, _ in training_pairs]

target_texts = ["<start> " + out + " <end>" for _, out in training_pairs]

In [ ]:
VOCAB_SIZE = 20000
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    filters='',
    oov_token="<unk>"
)

tokenizer.fit_on_texts(input_texts + target_texts)

In [ ]:
#Convertir a secuencias
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

In [ ]:
# TAMAÑO DE VOCABULARIO
vocab_size = len(tokenizer.word_index) + 1
print("vocab_size:", vocab_size)

vocab_size: 163473


In [ ]:
#PADDING
encoder_input = pad_sequences(input_sequences, maxlen=MAX_LEN, padding="post")

decoder_input = pad_sequences(target_sequences, maxlen=MAX_LEN, padding="post")

In [ ]:
#DECODER TARGET
decoder_target = np.zeros_like(decoder_input)

decoder_target[:, :-1] = decoder_input[:, 1:]

## Construccion del modelo

###ENCODER

In [ ]:
#Parametros del modelo
EMBEDDING_DIM = 200
LSTM_UNITS = 256
MAX_LEN = 25
VOCAB_SIZE = 20000


In [ ]:
#ENCODER
encoder_inputs = Input(shape=(MAX_LEN,))

In [ ]:
#EMBEDDING ENCODER
encoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    mask_zero=True
)(encoder_inputs)

In [ ]:
#LSTM del encoder:
encoder_lstm = Bidirectional(LSTM(
    LSTM_UNITS,
    return_state=True,
    return_sequences=True
))

encoder_outputs, fh, fc, bh, bc = encoder_lstm(encoder_embedding)

state_h = Concatenate()([fh, bh])
state_c = Concatenate()([fc, bc])

encoder_states = [state_h, state_c]

###DECODER

In [ ]:
#ENTRADA DECODER
decoder_inputs = Input(shape=(MAX_LEN,))
#EMBEDDING DECODER
decoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    mask_zero=True
)(decoder_inputs)

In [ ]:
#LSTM del decoder (inicializado con los estados del encoder):
decoder_lstm = LSTM(
    LSTM_UNITS*2,
    return_sequences=True,
    return_state=True,
    dropout=0.3,
    recurrent_dropout=0.3
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [ ]:
print("max encoder:", encoder_input.max())
print("max decoder:", decoder_input.max())
print("max target:", decoder_target.max())
print("VOCAB_SIZE:", VOCAB_SIZE)

max encoder: 19999
max decoder: 19999
max target: 19999
VOCAB_SIZE: 20000


In [ ]:
from tensorflow.keras.layers import AdditiveAttention

attention_layer = AdditiveAttention()

attention_output = attention_layer(
    [decoder_outputs, encoder_outputs]
)

In [ ]:
#Concatenar
decoder_concat = Concatenate(axis=-1)(
    [decoder_outputs, attention_output]
)

###CAPA DE SALIDA

In [ ]:
decoder_dense = Dense(
    VOCAB_SIZE,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

CORRECCION DE LINEA VOLVERA ENTRENAR

In [ ]:
from tensorflow.keras.optimizers import Adam

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0003,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False  # Output layer uses softmax, so from_logits should be False
    ),
    metrics=["accuracy"]
)

In [ ]:
# The decoder_target is already (num_samples, MAX_LEN) from xdl-YibKvOK4, just ensure it's int32.
# Remove the unnecessary reshape to (None, MAX_LEN, 1).
decoder_target = decoder_target.astype("int32")

In [ ]:
#acelerar computo
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
#entrenamiento
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=128,
    epochs=10,
    validation_split=0.1
)

Epoch 1/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 358s 68ms/step - accuracy: 0.2637 - loss: 5.0187 - val_accuracy: 0.2880 - val_loss: 4.6580
Epoch 2/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 351s 68ms/step - accuracy: 0.2952 - loss: 4.5337 - val_accuracy: 0.2990 - val_loss: 4.4777
Epoch 3/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 354s 68ms/step - accuracy: 0.3035 - loss: 4.3796 - val_accuracy: 0.3048 - val_loss: 4.3940
Epoch 4/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 358s 69ms/step - accuracy: 0.3084 - loss: 4.2803 - val_accuracy: 0.3082 - val_loss: 4.3450
Epoch 5/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 355s 69ms/step - accuracy: 0.3120 - loss: 4.2039 - val_accuracy: 0.3112 - val_loss: 4.3114
Epoch 6/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 356s 69ms/step - accuracy: 0.3150 - loss: 4.1406 - val_accuracy: 0.3128 - val_loss: 4.2894
Epoch 7/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 355s 69ms/step - accuracy: 0.3174 - loss: 4.0856 - val_accuracy: 0.3141 - val_loss: 4.2764
Epoch 8/10
5171/5171 ━━━━━━━━━━━━━━━━━━━━ 353s 68ms/step - accuracy: 

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 25, 200)   │  4,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 25)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ [(None, 25, 512), │    935,936 │ embedding[0][0],  │
│ (Bidirectional)     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 25, 200)   │  4,000,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 25, 512), │  1,460,224 │ embedding_1[0][0… │
│                     │ (None, 512),      │            │ concatenate[0][0… │
│                     │ (None, 512)]      │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 25, 20000) │ 10,260,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 61,968,482 (236.39 MB)

 Trainable params: 20,656,160 (78.80 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 41,312,322 (157.59 MB)

###Modelos encoder(inferencia)

In [ ]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

In [ ]:
#entrada decoder
decoder_state_input_h = Input(shape=(LSTM_UNITS*2,))
decoder_state_input_c = Input(shape=(LSTM_UNITS*2,))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

In [ ]:
#Reutilizar embedding y LSTM del decoder
decoder_inputs_single = Input(shape=(1,))

# Get the actual Embedding layer object for the decoder from the trained model
decoder_embedding_layer_inference = model.get_layer('embedding_1') # Corrected layer name

decoder_emb = decoder_embedding_layer_inference(decoder_inputs_single)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_emb,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]

decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
#Modelo final decoder
decoder_model = Model(
    [decoder_inputs_single] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [ ]:
#Dicccionario indice
index_word = tokenizer.index_word
word_index = tokenizer.word_index

**Top-k sampling** limita las opciones del modelo a las k palabras más probables y luego hace sampling solo entre ellas.
Esto evita que el modelo escoja palabras muy improbables (ruido) pero mantiene diversidad.

In [ ]:
def top_p_sampling(probs, p=0.9, temperature=0.6):

    probs = probs.astype("float64")

    # aplicar temperatura
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)

    # ordenar probabilidades
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]

    # acumulado
    cumulative_probs = np.cumsum(sorted_probs)

    # mantener tokens hasta cubrir p
    cutoff = cumulative_probs > p
    cutoff_index = np.argmax(cutoff)

    candidate_indices = sorted_indices[:cutoff_index + 1]
    candidate_probs = probs[candidate_indices]

    candidate_probs = candidate_probs / np.sum(candidate_probs)

    token_index = np.random.choice(candidate_indices, p=candidate_probs)

    return token_index

def sample_with_temperature(preds, temperature=0.7):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-9) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)

    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

# Define start_token_id here, assuming word_index is available
start_token_id = tokenizer.word_index.get("<start>")

def generate_response(sentence, max_len=MAX_LEN, temperature=0.7):

    sentence = clean_text(sentence)

    seq = tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding="post")

    states = encoder_model.predict(seq, verbose=0)
    target_seq = np.array([[start_token_id]])

    response_tokens = []
    unk_token_id = tokenizer.word_index.get("<unk>", None)

    for _ in range(max_len):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        probs = output_tokens[0, -1, :].astype("float64")

        # bloquear tokens indeseados
        probs[0] = 0
        if unk_token_id is not None and unk_token_id < len(probs):
            probs[unk_token_id] = 0
        if start_token_id < len(probs):
            probs[start_token_id] = 0

        # top-k sampling
        token_index = top_p_sampling(probs, p=0.9, temperature=0.6)

        word = tokenizer.index_word.get(token_index, "")

        if word == "<end>" or word == "":
            break

        response_tokens.append(word)

        target_seq = np.array([[token_index]])
        states = [h, c]

    if not response_tokens:
        return "i don't know."

    return " ".join(response_tokens)


In [ ]:
print(generate_response("hello"))
print(generate_response("how are you"))
print(generate_response("who are you"))

hello bill.
i don't know you were a terrible thing i know you're going to make it happen.
you have to be the one who do i want to go back to the head?


In [ ]:
generate_response("do you want play?")

'no.'

In [ ]:
generate_response("i need help")

"i'm going to tell you what you want to do this."

In [ ]:
generate_response("are you human?")

'no.'

In [ ]:
generate_response("are you a man or a woman?")

"no i didn't think so."

In [ ]:
generate_response("hi, geaorgi")

'hi walter.'

In [ ]:
generate_response("what happen?")

'she gave her the wrong number.'

In [ ]:
generate_response("hi")

"i'm a good man"

In [ ]:
generate_response("tell me something")

"you can't go to the hotel..."

###Guardar modelo para no volver a reentrenar

In [ ]:
model.save("chatbot_seq2seq.h5")
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
from google.colab import files

files.download("chatbot_seq2seq.h5")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
encoder_model.save("encoder_model.h5")
decoder_model.save("decoder_model.h5")
files.download("encoder_model.h5")
files.download("decoder_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>